In [78]:
import pandas as pd
from sqlalchemy import create_engine

In [79]:
def trim_text(col):
    return col.astype(str).str.strip() if col.dtype == 'str' else col

In [80]:
BASE_PATH = "C:/Users/Jerry/Desktop/Analysis_projects_TSQL/E-commerce Transactions + Clickstream/dataset"
files = ["customers", "products", "events", "orders", "order_items", "sessions", "reviews"]
dataframes = {file: pd.read_csv(f"{BASE_PATH}/{file}.csv").apply(trim_text) for file in files}


## Transformations


### Renaming columns

In [81]:

#customers (marketing_opt_in to receives_promos)
dataframes["customers"].rename(columns={"marketing_opt_in": "receives_promos"}, inplace=True)

#products (price_usd, cost_usd, margin_usd to price, cost, margin)
dataframes["products"].rename(columns={"price_usd": "price", "cost_usd": "cost", "margin_usd": "margin"}, inplace=True)

#orders (discount_pct, subtotal_usd, total_usd to discount_percentage, subtotal, total)
dataframes["orders"].rename(columns={"discount_pct": "discount_percentage", "subtotal_usd": "subtotal", "total_usd": "total"}, inplace=True)

#order_items (unit_price_usd, line_total_usd, to unit_price, total_line)
dataframes["order_items"].rename(columns={"unit_price_usd": "unit_price", "line_total_usd": "total_line"}, inplace=True)

#events (qty, discount_pct, amount_usd to quantity, discount_percentage, amount)
dataframes["events"].rename(columns={"qty": "quantity", "discount_pct": "discount_percentage", "amount_usd": "amount"}, inplace=True)



### Handling duplicates


##### customers

In [82]:
#Run the following code to check the data types of each column and its non-null count in the customers dataframe
dataframes['customers'].info()
#Run the following code to check the number of unique values in each column of the customers dataframe
dataframes['customers'].nunique()

#we need to change the data type of the signup_date column in the customers dataframe to datetime
dataframes['customers']['signup_date'] = pd.to_datetime(dataframes['customers']['signup_date'], errors='coerce')

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   customer_id      20000 non-null  int64
 1   name             20000 non-null  str  
 2   email            20000 non-null  str  
 3   country          20000 non-null  str  
 4   age              20000 non-null  int64
 5   signup_date      20000 non-null  str  
 6   receives_promos  20000 non-null  bool 
dtypes: bool(1), int64(2), str(4)
memory usage: 957.2 KB



##### products

In [83]:
#Run the following code to check the data types of each column and its non-null count in the products dataframe
dataframes['products'].info()
#Run the following code to check the number of unique values in each column of the products dataframe
dataframes['products'].nunique()

<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   product_id  1197 non-null   int64  
 1   category    1197 non-null   str    
 2   name        1197 non-null   str    
 3   price       1197 non-null   float64
 4   cost        1197 non-null   float64
 5   margin      1197 non-null   float64
dtypes: float64(3), int64(1), str(2)
memory usage: 56.2 KB


product_id    1197
category         7
name          1197
price         1161
cost          1157
margin        1096
dtype: int64


##### sessions

In [84]:
#Run the following code to check the data types of each column and its non-null count in the sessions dataframe
dataframes['sessions'].info()
#Run the following code to check the number of unique values in each column of the sessions dataframe
#dataframes['sessions'].nunique()

#we need to change the data type of the start_time column in the sessions dataframe to datetime
dataframes['sessions']['start_time'] = pd.to_datetime(dataframes['sessions']['start_time'], errors='coerce')

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   session_id   120000 non-null  int64
 1   customer_id  120000 non-null  int64
 2   start_time   120000 non-null  str  
 3   device       120000 non-null  str  
 4   source       120000 non-null  str  
 5   country      120000 non-null  str  
dtypes: int64(2), str(4)
memory usage: 5.5 MB



##### orders

In [85]:
#Run the following code to check the data types of each column and its non-null count in the orders dataframe
dataframes['orders'].info()
#Run the following code to check the number of unique values in each column of the orders dataframe
#dataframes['orders'].nunique()

#we need to change the data type of the order_time column in the orders dataframe to datetime
dataframes['orders']['order_time'] = pd.to_datetime(dataframes['orders']['order_time'], errors='coerce')

<class 'pandas.DataFrame'>
RangeIndex: 33580 entries, 0 to 33579
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   order_id             33580 non-null  int64  
 1   customer_id          33580 non-null  int64  
 2   order_time           33580 non-null  str    
 3   payment_method       33580 non-null  str    
 4   discount_percentage  33580 non-null  int64  
 5   subtotal             33580 non-null  float64
 6   total                33580 non-null  float64
 7   country              33580 non-null  str    
 8   device               33580 non-null  str    
 9   source               33580 non-null  str    
dtypes: float64(2), int64(3), str(5)
memory usage: 2.6 MB



##### order_items

In [86]:
#Run the following code to check the data types of each column and its non-null count in the order_items dataframe
dataframes['order_items'].info()
#Run the following code to check the number of unique values in each column of the order_items dataframe
dataframes['order_items'].nunique()
#Here we need to make sure the combination of order_id and product_id is unique in the order_items dataframe.
#If not, we need to investigate why there are duplicates and how to handle them.
dataframes['order_items'].duplicated(subset=['order_id', 'product_id']).sum()

#We decided to aggregate the order_items dataframe by order_id and product_id, summing the quantity and total_line, and taking the mean of unit_price.
#This way, we can ensure that each combination of order_id and product_id is unique in the order_items dataframe.

dataframes['order_items'] = dataframes['order_items'].groupby(['order_id', 'product_id'], as_index=False).agg({
    'quantity': 'sum',
    'unit_price': 'mean',
    'total_line': 'sum'
})

#After aggregating the order_items dataframe, we can check again for duplicates to ensure that the combination of order_id and product_id is now unique.
dataframes['order_items'].duplicated(subset=['order_id', 'product_id']).sum()

<class 'pandas.DataFrame'>
RangeIndex: 59163 entries, 0 to 59162
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    59163 non-null  int64  
 1   product_id  59163 non-null  int64  
 2   unit_price  59163 non-null  float64
 3   quantity    59163 non-null  int64  
 4   total_line  59163 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 2.3 MB


np.int64(0)


##### reviews

In [87]:
#Run the following code to check the data types of each column and its non-null count in the reviews dataframe
dataframes['reviews'].info()
#Run the following code to check the number of unique values in each column of the reviews dataframe
#dataframes['reviews'].nunique()

#we need to change the data type of the review_time column in the reviews dataframe to datetime
dataframes['reviews']['review_time'] = pd.to_datetime(dataframes['reviews']['review_time'], errors='coerce')

<class 'pandas.DataFrame'>
RangeIndex: 10780 entries, 0 to 10779
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   review_id    10780 non-null  int64
 1   order_id     10780 non-null  int64
 2   product_id   10780 non-null  int64
 3   rating       10780 non-null  int64
 4   review_text  10780 non-null  str  
 5   review_time  10780 non-null  str  
dtypes: int64(4), str(2)
memory usage: 505.4 KB



##### events

In [88]:
#Run the following code to check the data types of each column and its non-null count in the events dataframe
dataframes['events'].info()
#Run the following code to check the number of unique values in each column of the events dataframe
#dataframes['events'].nunique()
#Here we see that our main four columns(event_id, session_id, timestamp, event_type) are not null which is good.

#we need to change the data type of the timestamp column in the events dataframe to datetime
dataframes['events']['timestamp'] = pd.to_datetime(dataframes['events']['timestamp'], errors='coerce')

<class 'pandas.DataFrame'>
RangeIndex: 760958 entries, 0 to 760957
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   event_id             760958 non-null  int64  
 1   session_id           760958 non-null  int64  
 2   timestamp            760958 non-null  str    
 3   event_type           760958 non-null  str    
 4   product_id           682469 non-null  float64
 5   quantity             143126 non-null  float64
 6   cart_size            44909 non-null   float64
 7   payment              33580 non-null   str    
 8   discount_percentage  33580 non-null   float64
 9   amount               33580 non-null   float64
dtypes: float64(5), int64(2), str(3)
memory usage: 58.1 MB



## Connecting to PostgreSQL

In [89]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

# Build connection string dynamically
db_url = f"postgresql+psycopg://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"

engine = create_engine(db_url)


### Writing to the tables

In [ ]:
# The order in which we insert data is important due to foreign key constraints.
# We need to insert data into the parent tables first, followed by the child tables.
# The order of insertion should be: customers, products, sessions, orders, order_items, reviews, events.
tables = ['customers', 'products', 'sessions', 'orders', 'order_items', 'reviews', 'events']

#Use engine.begin() to start a transaction context
with engine.begin() as conn:
    for table in tables:
        # Pass the connection 'conn' instead of 'engine'
        dataframes[table].to_sql(table, con=conn, if_exists='append', index=False)
        print(f"Data inserted into {table} table.")

Data inserted into customers table.
Data inserted into products table.
Data inserted into sessions table.
Data inserted into orders table.
Data inserted into order_items table.
Data inserted into reviews table.


error ignored terminating <psycopg.Pipeline [INERROR, pipeline=ON] (host=localhost user=postgres database=E-commerce_Transactions_Clickstream) at 0x152bd2d9590>: pipeline aborted


DatabaseError: Execution failed on sql 'INSERT INTO events (event_id, session_id, timestamp, event_type, product_id, quantity, cart_size, payment, discount_percentage, amount) VALUES (:event_id, :session_id, :timestamp, :event_type, :product_id, :quantity, :cart_size, :payment, :discount_percentage, :amount)': (psycopg.errors.UndefinedColumn) column "amount" of relation "events" does not exist
LINE 1: ...uantity, cart_size, payment, discount_percentage, amount) VA...
                                                             ^
[SQL: INSERT INTO events (event_id, session_id, timestamp, event_type, product_id, quantity, cart_size, payment, discount_percentage, amount) VALUES (%(event_id)s::BIGINT, %(session_id)s::BIGINT, %(timestamp)s::TIMESTAMP WITHOUT TIME ZONE, %(event_type)s::VARCHAR, %(product_id)s, %(quantity)s, %(cart_size)s, %(payment)s::VARCHAR, %(discount_percentage)s, %(amount)s)]
[parameters: [{'event_id': 1, 'session_id': 1, 'timestamp': datetime.datetime(2021, 12, 27, 0, 

### Future updates
Checking foreign key dependencies in pandas(See if a foreign key kkey column exist in its primary key before inserting)
